In [ ]:
'''This part contains the code for my first experiment in my paper.'''

In [ ]:
save_interval = 100
start_epoch = 1
current_model_checkpoint = ""

num_epochs = 3000

def train(content_weight = 1.0, style_weight=10.0, tv_weight=1e-5, lr=1e-3):
    content_nodes = ['relu_3_3']
    style_nodes = ['relu_1_2', 'relu_2_2', 'relu_3_3', 'relu_4_3']
    return_nodes = {3: 'relu_1_2',
                    8: 'relu_2_2',
                    15: 'relu_3_3',
                    22: 'relu_4_3'}

    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
    for param in vgg.parameters():
        param.requires_grad = False
    loss_network = create_feature_extractor(vgg, return_nodes)
    loss_network = loss_network.to(device)


    # network
    model = StyleTransferNetwork()
    if current_model_checkpoint:
      model.load_state_dict(torch.load(current_model_checkpoint))

    model = model.to(device)
    model.train()
    optimizer = Adam(model.parameters(), lr=lr)
    style_iterator = iter(style_dataloader)

    print("Start training...")
    #-------------------

    # New -- Epoch Loop
    for epoch in range(start_epoch, 1 + num_epochs):

        print(f'Epoch {epoch} / {num_epochs}')
        '''
        These variables will keep track of the cumulative loss over all the batches in the content_dataset in this epoch.
        Then in the next epoch, they'll be re-initialized to 0.
        '''
        cumulative_total_loss = 0.0
        cumulative_style_loss = 0.0
        cumulative_content_loss = 0.0

        # New -- Batch Loop
        for batch_idx, (content_images, content_images_idxs) in enumerate(content_dataloader):
          print(f'Batch {batch_idx} / {len(content_dataloader)}')
          print('verify content', content_images.shape, content_images_idxs)

          try:
            style_images, style_id = next(style_iterator) # style_batch = 1

          except StopIteration:
            style_iterator = iter(style_dataloader)
            style_images, style_id = next(style_iterator)


          style_codes = torch.zeros(content_batch_size, NUM_STYLE, 1)
          style_codes[:, style_id, :] = 1 # Set the corresponding style_id to 1 for all items in the batch
          style_codes = style_codes.to(device)

          content_images = content_images.to(device)
          style_images = style_images.to(device)

          output_images = model(content_images, style_codes)

          output_features = loss_network(output_images)
          content_features = loss_network(content_images)

          style_features = loss_network(style_images)

          style_loss = calc_style_loss_expand(output_features, style_features, style_nodes)
          # equivalent to calc_style_loss_custom

          content_loss = calc_content_loss (output_features, content_features, content_nodes)
          tv_loss = calc_tv_loss(output_images)

          total_loss = content_loss * content_weight + style_loss * style_weight + tv_loss * tv_weight

          # New
          '''
          Keeps track of accumulating losses over all the batches in this epoch.
          In the next epoch, these values will be re-initialized to 0.
          '''
          cumulative_total_loss += total_loss.item()
          cumulative_style_loss += style_loss.item()
          cumulative_content_loss += content_loss.item()


          optimizer.zero_grad()
          total_loss.backward()
          optimizer.step()



        # This is done at the level of per epoch (not per batch!)
        total_losses.append(cumulative_total_loss / len(content_dataloader))
        style_losses.append(cumulative_style_loss /  len(content_dataloader))
        content_losses.append(cumulative_content_loss / len(content_dataloader))

        global loss_path
        save_losses(loss_path)

        if (epoch % save_interval == 0):
          print(f'Epoch {epoch} / {num_epochs} | Style Loss: {style_losses[-1]:.4f} | Content Loss: {content_losses[-1]:.4f}')
          path = f"/content/drive/MyDrive/Movie_Project/Tuning_style_weights/content_1_style_10000/Non-avg_gram/Model/Model_epoch_{epoch}.pth"
          # The path is customized to the link where I want to save the output models'''

          torch.save(model.state_dict(), path)

    return model